# 0.2 锻炼代码逻辑思维

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/00_guide/0.2_code_thinking.ipynb)


很多人"代码能力一般"，问题往往不在语法，而在下面三件事：

- 拿到一个需求，**不知道从哪里下手**；
- 写到一半，**脑子里的数据流断了**，不知道这个变量是什么形状、什么含义；
- 代码跑起来结果不对，**不知道怎么定位**。

这一课给你四个习惯。它们看起来朴素，但所有能"从零写模型"的人都在用。后面每一课的练习都会刻意训练它们。

---

## 习惯一：先写"契约"，再写实现

写任何函数之前，先回答三个问题，并把答案**写成注释**：

1. **输入**是什么？类型、形状、取值范围、含义。
2. **输出**是什么？同上。
3. **一句话**说明它做了什么。

例如，VLA 里最常见的"动作归一化"：

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import numpy as np

def normalize_actions(actions, mean, std):
    """把动作减均值除以标准差，让每个维度都差不多在 [-1, 1] 附近。

    输入:
        actions: float 数组 (T, D)，T 个时间步，每步 D 维动作（比如 7 个关节角）
        mean, std: float 数组 (D,)，在整个训练集上统计出来的每个维度的均值、标准差
    输出:
        float 数组 (T, D)，归一化之后的动作
    """
    return (actions - mean) / (std + 1e-8)   # +1e-8 防止某个维度 std=0 时除以 0

acts = np.array([[0.1, 2.0], [0.3, 4.0], [0.2, 3.0]])
print(normalize_actions(acts, acts.mean(0), acts.std(0)))

写契约的过程会**逼你想清楚问题**。很多 bug 在这一步就避免了。比如你会意识到"std 可能为 0"。

---

## 习惯二：时刻知道每个张量的形状

深度学习代码 80% 的 bug 是**形状错了**或者**维度用错了**。规则很简单：

- 每一个张量变量，在旁边注释它的形状：`# (B, T, D)`；
- 关键位置用 `assert` 把形状钉死，错了立刻报错，而不是在 100 行以后莫名其妙地出错。

常用字母约定（本项目全程使用）：

| 字母 | 含义 | 例子 |
|---|---|---|
| `B` | batch size，一批有几个样本 | 64 |
| `T` / `H` | 时间步数 / 动作块长度（action horizon） | 50 |
| `L` / `N` | token 序列长度 | 256 + 48 |
| `D` | 特征维度（hidden size） | 1024 |
| `C, H, W` | 图像通道、高、宽 | 3, 224, 224 |

In [ ]:
import torch

B, T, D = 4, 10, 7
actions = torch.randn(B, T, D)          # (B, T, D)：一批动作块
first_step = actions[:, 0]              # (B, D)：每个样本的第一步动作
mean_over_time = actions.mean(dim=1)    # (B, D)：在时间维度上求平均
assert mean_over_time.shape == (B, D)
print(first_step.shape, mean_over_time.shape)

### 练习 1：修复形状 bug

下面的函数想计算"每个样本、每个时间步的动作大小（L2 范数）"，输出应该是 `(B, T)`。它有一个 bug，请修好它。

In [ ]:
def action_magnitude(actions):
    """actions: (B, T, D) -> 返回 (B, T)，每一步动作向量的 L2 范数"""
    # TODO: 下面这一行的 dim 用错了，请改正
    return actions.norm(dim=1)

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def action_magnitude(actions):
    """actions: (B, T, D) -> 返回 (B, T)，每一步动作向量的 L2 范数"""
    # 范数是对"每个动作向量内部"的 D 个分量求的，所以要在最后一维 (dim=-1) 上做
    return actions.norm(dim=-1)

In [ ]:
# ✅ 自测
a = torch.randn(4, 10, 7)
out = action_magnitude(a)
assert out.shape == (4, 10), f"形状应为 (4, 10)，实际是 {tuple(out.shape)}"
assert torch.allclose(out[1, 3], a[1, 3].pow(2).sum().sqrt())
print("通过 ✅")

> 思路：先问自己"范数是对哪几个数算的？"。答案是一个动作向量的 7 个分量，它们在最后一维。所以 `dim=-1`。**先想清楚含义，再选维度**。

---

## 习惯三：从小到大，从慢到快

不要一上来就写"完整的训练系统"。正确的顺序是：

1. **单个样本、for 循环**版本：最慢，但最容易写对；
2. **批量化（向量化）**版本：快，用第 1 步的结果检查它；
3. 先在**一个 batch 上过拟合**：loss 能降到接近 0，说明模型和数据流没问题；
4. 再上完整数据、调超参。

第 3 步极其重要。一个模型连一个 batch 都记不住，在全量数据上也不可能学好。这时候应该查 bug，而不是调学习率。

### 练习 2：把 for 循环改成向量化

下面是计算"每个样本到目标点距离"的循环版本。请写出不用 for 循环的版本。

In [ ]:
import numpy as np

def dist_loop(pos, goal):
    """pos: (N, 2)，goal: (2,) -> (N,)"""
    out = np.zeros(len(pos))
    for i in range(len(pos)):
        out[i] = np.sqrt((pos[i, 0] - goal[0]) ** 2 + (pos[i, 1] - goal[1]) ** 2)
    return out

def dist_vectorized(pos, goal):
    """pos: (N, 2)，goal: (2,) -> (N,)"""
    # TODO: 利用广播 pos - goal，然后在最后一维上求范数
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def dist_vectorized(pos, goal):
    """pos: (N, 2)，goal: (2,) -> (N,)"""
    diff = pos - goal               # (N, 2) - (2,) -> 广播成 (N, 2)
    return np.linalg.norm(diff, axis=-1)   # (N,)

In [ ]:
# ✅ 自测：用慢但可靠的循环版本检查快的版本（这就是"从慢到快"）
pos = np.random.randn(1000, 2); goal = np.array([0.3, -0.2])
assert np.allclose(dist_loop(pos, goal), dist_vectorized(pos, goal))
print("通过 ✅")

---

## 习惯四：读源码时追"一条数据"

读大型代码库（openpi、FluxVLA、RLinf）时，不要从第一个文件读到最后一个文件。正确做法是：

1. 找到**入口**（训练脚本的 `main`，或者模型的 `forward`）；
2. 拿**一条数据**，跟着它走：它在每个函数里变成了什么形状、什么含义；
3. 边追边**画数据流图**：方框是函数，箭头上标形状。

以第 7 章要精读的 π0（openpi 的 `pi0_pytorch.py`）训练前向为例，画出来的图大概是：

```text
图像 (B,3,224,224) ──SigLIP──> 图像 token (B,256,2048) ─┐
文本 id (B,48) ──embedding──> 文本 token (B,48,2048) ───┴─> prefix (B,304,2048)
                                                                    │
带噪动作 x_t (B,50,32) ──线性层──> 动作 token (B,50,1024) ── suffix  │
时间 t (B,) ──正弦编码+MLP──> 时间条件 ───────────────────────┘     │
                                                                    ▼
                         [VLM 与 动作专家 共享注意力的 Transformer]
                                                                    ▼
                       动作 token 输出 (B,50,1024) ──线性层──> 预测速度 v_t (B,50,32)
                                                                    ▼
                                       loss = MSE(v_t, 噪声 - 真实动作)
```

能画出这样一张图，说明你已经读懂了这个模型 70%。剩下的 30% 是细节（mask、位置编码、KV cache），后面会逐个讲。

### 练习 3：写"契约 + 实现"

实现函数 `chunk_actions(actions, H)`：把一条轨迹的动作 `(T, D)` 切成"动作块"。对每个时间步 `t`，取出 `actions[t : t+H]`。如果超出轨迹末尾，就用**最后一个动作**补齐。输出 `(T, H, D)`。

这是第 5 章 ACT 和第 7 章 VLA 都要用到的"动作分块（action chunking）"数据处理。

In [ ]:
def chunk_actions(actions, H):
    """
    输入: actions (T, D)；H 动作块长度
    输出: (T, H, D)，第 t 行是从 t 开始的 H 个动作，越界部分用 actions[-1] 填充
    """
    # TODO: 提示：先把 actions 末尾补 H-1 个 actions[-1]，再用 for 循环（或高级索引）切片
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def chunk_actions(actions, H):
    """
    输入: actions (T, D)；H 动作块长度
    输出: (T, H, D)，第 t 行是从 t 开始的 H 个动作，越界部分用 actions[-1] 填充
    """
    T = actions.shape[0]
    pad = np.repeat(actions[-1:], H - 1, axis=0)       # (H-1, D)：重复最后一个动作
    padded = np.concatenate([actions, pad], axis=0)    # (T+H-1, D)
    idx = np.arange(T)[:, None] + np.arange(H)[None, :]  # (T, H)：第 t 行是 [t, t+1, ..., t+H-1]
    return padded[idx]                                 # 高级索引 -> (T, H, D)

In [ ]:
# ✅ 自测
acts = np.arange(10, dtype=float).reshape(5, 2)   # 5 步，每步 2 维
c = chunk_actions(acts, 3)
assert c.shape == (5, 3, 2)
assert np.allclose(c[0], acts[0:3])
assert np.allclose(c[4], [acts[4], acts[4], acts[4]])   # 最后一步：全部用最后一个动作补齐
assert np.allclose(c[3], [acts[3], acts[4], acts[4]])
print("通过 ✅")

---

## 小结

| 习惯 | 一句话 |
|---|---|
| 先写契约 | 动手前先写清楚输入、输出、形状 |
| 时刻标注形状 | 每个张量都注释形状，关键处 `assert` |
| 从小到大 | 循环版检查向量版；先过拟合一个 batch |
| 追一条数据 | 读源码时跟着一条数据走，画数据流图 |

**思考题**：回忆一下你最近遇到的一个 bug，它属于上面哪个习惯没做好？

下一章：[第 1 章 预备知识](../01_prerequisites/README.md)。